# Lab 6.1 &mdash; Hello Qwen &mdash; the Sandbox Model

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 15 min &nbsp;|&nbsp; **Day 2 &middot; Module 6 &mdash; Agentic RAG**

### What you'll do
- Send your first call to the sandbox model and read what comes back
- Decide what belongs in the <strong>system</strong> role and what belongs in the <strong>human</strong> role
- Choose the two settings that decide whether today costs you cents or dollars
- Watch the model's own reasoning appear in the token bill &mdash; and switch it off

> **How this lab works.** You write real LangChain code. Fill every `BLANK`, then run the
> **Self-check** cell under each section &mdash; those check the *objects you built* (a
> `Document`, a Chroma collection, a retriever, a chain), so they are deterministic and do not
> depend on the chat model. Cells marked **Run it for real** put your code in front of the
> sandbox model; that is the part worth watching. The score line is feedback, not a grade.

> **Two different models are in play, and only one of them is billed.** The **chat model**
> (`qwen36-35b-a3b-lab`) answers questions and is reached over the gateway. The **embedding
> model** (`all-MiniLM-L6-v2`, 384 dimensions) turns text into vectors and runs on this pod's
> own CPU &mdash; no key, no gateway, no tokens. Keeping them straight is most of Module 6.

> **Nothing to install and no key to register.** The model is already wired into this
> sandbox. If a live cell says it is not configured, run `env | grep -i llm` in a
> terminal and export the two values it names.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap, warnings
from typing import Any, Callable

warnings.filterwarnings("ignore")     # sentence-transformers is chatty on first import

WORK = os.path.join("/tmp", "awmas-lab-6-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the CHAT model: qwen, through the sandbox gateway -------------------
# Already configured -- nothing to install, no key to register. Read from the
# environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens. Thinking is off by default here because you will make a lot of calls today;
# pass think=True to any call below to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Chat model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- the EMBEDDING model: local, free, nothing to configure --------------
# all-MiniLM-L6-v2, 384 dimensions. It runs on this pod's CPU and has nothing to do with
# the chat model above: no gateway, no key, no tokens billed. The cache is already warm
# in your sandbox, so the first call is a second or two, not a download.
#
# It is reached through onnxruntime rather than torch, and that is a measured choice
# rather than a taste: same model, same vectors, ~170 MB of memory instead of ~840. Your
# whole sandbox has 2.5 GB for every notebook you leave open, and a kernel you have
# forgotten about is still holding its share.
from langchain_core.embeddings import Embeddings

class MiniLMEmbeddings(Embeddings):
    """all-MiniLM-L6-v2 behind LangChain's Embeddings interface.

    Two methods is the whole contract -- which is why a store, a splitter and a chain
    never need to know which model is underneath, or what runtime it uses."""

    def __init__(self):
        from chromadb.utils.embedding_functions import ONNXMiniLM_L6_V2
        self._fn = ONNXMiniLM_L6_V2()

    def embed_documents(self, texts: list) -> list:
        return [[float(x) for x in v] for v in self._fn(list(texts))]

    def embed_query(self, text: str) -> list:
        return [float(x) for x in self._fn([text])[0]]


_emb_cache = {}
def get_embeddings():
    """The embedding model, built once per kernel."""
    if "model" not in _emb_cache:
        _emb_cache["model"] = MiniLMEmbeddings()
    return _emb_cache["model"]

print("work dir   :", WORK)
print("chat model :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

## Concept

Every call in this module is the same three things:

| | |
|---|---|
| **a system message** | the standing instruction &mdash; who the model is, what shape the answer takes |
| **a human message** | the thing you actually want to know, which changes every call |
| **settings** | `temperature`, and whether the model reasons before answering |

The chat model is reached over the sandbox gateway and every token is billed against your
own daily budget. The embedding model you meet in Lab 6.2 is not: it runs here, on this pod.

## Section 1 &mdash; Two roles, one call

The instruction and the question go in different places. Put the question in the system
message and it becomes part of the model's standing character &mdash; which is exactly the
bug you get when a prompt template is built by string concatenation.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

INSTRUCTION = "You are a concise assistant. Answer in at most two sentences."

def build_messages(question: str) -> list:
    """One standing instruction, one question. Which content goes in which role?"""
    return [
        SystemMessage(content=INSTRUCTION),
        HumanMessage(content=question),
    ]

In [ ]:
# --- Self-check: Section 1   (message objects only -- no call yet)
Q = "What is retrieval-augmented generation?"

check("build_messages returns exactly two messages",
      lambda: len(build_messages(Q)) == 2)
check("the first is a SystemMessage and the second a HumanMessage",
      lambda: isinstance(build_messages(Q)[0], SystemMessage)
              and isinstance(build_messages(Q)[1], HumanMessage))
check("the standing instruction is in the SYSTEM message",
      lambda: build_messages(Q)[0].content == INSTRUCTION)
check("the question is in the HUMAN message, and only there",
      lambda: build_messages(Q)[1].content == Q and Q not in build_messages(Q)[0].content,
      "a question baked into the system message becomes part of every later turn")

In [ ]:
# --- Run it for real -------------------------------------------------------
def first_call():
    llm = get_llm()
    reply = llm.invoke(build_messages("What is retrieval-augmented generation?"))
    print(reply.content.strip())
    print("\ntokens:", reply.usage_metadata)

if llm_ready():
    guard(first_call)

## Section 2 &mdash; The two settings that decide the bill

`temperature` decides how much the model varies between identical calls. Reasoning
(&ldquo;thinking&rdquo;) decides how much it writes before it answers &mdash; and that
preamble is billed as completion tokens exactly like the answer is.

Both are decisions, not defaults. You are about to run a few hundred calls today.

In [ ]:
def grading_temperature() -> float:
    """You are about to compare two runs of the same question and score the difference.
    Which temperature makes that comparison mean anything?"""
    return 0.0

def thinking_for_labs() -> bool:
    """Reasoning is billed as completion tokens. Across a day of small factual lookups
    against a handbook, should the labs leave it on?"""
    return False

def body_for(think: bool) -> dict:
    """The extra_body the gateway wants. Given -- it is a dict shape, not a decision."""
    return {"chat_template_kwargs": {"enable_thinking": think}}

In [ ]:
# --- Self-check: Section 2   (values and dict shapes -- still no call)
check("a run you intend to compare is deterministic",
      lambda: grading_temperature() == 0.0,
      "anything above 0 and the difference you measure might just be sampling")
check("thinking is OFF for the labs",
      lambda: thinking_for_labs() is False,
      "it is the right default for a few hundred handbook lookups, not a universal one")
check("body_for builds what the gateway expects",
      lambda: body_for(False) == {"chat_template_kwargs": {"enable_thinking": False}})
check("and it matches the NO_THINK the setup cell already defined",
      lambda: body_for(thinking_for_labs()) == NO_THINK)

In [ ]:
# --- Run it for real -------------------------------------------------------
# The same question twice: once with reasoning off, once with it on. Watch the
# completion-token count, not the answer.
def thinking_costs_what():
    q = "A payment of 900,000 needs approval. Who approves it? Answer in one line."
    for think in (False, True):
        t0 = time.time()
        reply = get_llm(temperature=grading_temperature(), think=think).invoke(q)
        u = reply.usage_metadata or {}
        print(f"thinking={str(think):5}  completion={u.get('output_tokens', '?'):>5}  "
              f"total={u.get('total_tokens', '?'):>5}  {time.time() - t0:.1f}s")

if llm_ready():
    guard(thinking_costs_what)

In [ ]:
score()

## Your turn

1. Move the instruction into the human message instead, joined to the question with a
   newline. Does the answer change? Now ask a *second* question on the same model object
   and notice what the system message would have done that this does not.
2. Run the same question five times at `temperature=0.8` and five times at `0.0`. Count the
   distinct answers. That number is why every measurement in this course pins temperature.
3. `reply.usage_metadata` also carries `input_tokens`. Add a long system message and watch
   which half of the bill grows. That is the number Module 6 is about to make you pay
   attention to, because retrieved context lands in exactly that half.